# LLM4PatchCorrect — All Datasets
Runs LLM4PatchCorrect on Wang, RepairLLaMA, D4C, SRepair, PRISM.

**Run cells in order. Cell 1 causes a kernel restart — that is normal.**

## Cell 1: Install condacolab
⚠️ Kernel will restart automatically after this cell. That is normal — continue from Cell 2.

In [ ]:
!pip install condacolab -q
import condacolab
condacolab.install()


📢 Announcement 📢
condacolab==0.2 will be released soon! Try it with:

    !pip install -q https://github.com/conda-incubator/condacolab/archive/main.zip
    import condacolab
    condacolab.install()

0.2.x introduces a new installation method based on Pixi, with customizable Python versions.
This may be breaking for your workflow. If that's the case, please report it at
https://github.com/conda-incubator/condacolab and pin your `pip install` command to
condacolab==0.1 as a workaround.

⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/26.3.2-3/Miniforge3-26.3.2-3-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:07
🔁 Restarting kernel...


## Cell 2: Mount Drive and create conda environment
Run this after the kernel restarts.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!conda create -n llm4 python=3.10 -y -q
print('Conda env created!')

Mounted at /content/drive
Retrieving notices: ...working... done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: ...working... done

## Package Plan ##

  environment location: /usr/local/envs/llm4

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    bzip2-1.0.8                |      hda65f42_10         252 KB  conda-forge
    ca-certificates-2026.7.22  |       hbd8a1cb_0         129 KB  conda-forge
    ld_impl_linux-64-2.46.1    |default_hbd61a6d_103         725 KB  conda-forge
    libexpat-2.8.5             |       hd2095e1_0          76 KB  conda-forge
    libffi-3.7.0               |       h81df57d_1          66 KB  conda-forge
    libgcc-16.2.0              |       ha9f2e26_7         1.0 MB  conda-forge
    libgomp-16.2.0             |       he0feb66_7         626 KB  conda-forge
    liblzma-5.8.3              |       hb03c66

## Cell 3: Install packages in conda env

In [ ]:
!conda run -n llm4 pip install numpy==1.24.0 -q
!conda run -n llm4 pip install torch==2.1.0 --index-url https://download.pytorch.org/whl/cu121 -q
!conda run -n llm4 pip install setuptools -q
!conda run -n llm4 pip install safetensors -q
!conda run -n llm4 pip install sentencepiece torchvision -q
!conda run -n llm4 pip install tokenizers==0.13.3 -q
!conda run -n llm4 pip install huggingface-hub==0.14.1 -q
!conda run -n llm4 pip install transformers==4.30.0 --no-deps -q
!conda run -n llm4 pip install accelerate==0.27.0 -q
!conda run -n llm4 pip install bitsandbytes -q
!conda run -n llm4 pip install deepspeed==0.9.5 -q
!conda run -n llm4 pip install pandas==1.5.3 scikit-learn nltk Pillow requests tqdm loguru -q
!conda run -n llm4 pip install sentence-transformers --no-deps -q

!conda run -n llm4 python -c "import numpy, torch, transformers, accelerate, bitsandbytes; print('numpy:', numpy.__version__); print('torch:', torch.__version__); print('transformers:', transformers.__version__); print('accelerate:', accelerate.__version__); print('bitsandbytes:', bitsandbytes.__version__); print('All OK!')"

numpy: 1.24.0
torch: 2.14.1+cu130
transformers: 4.30.0
accelerate: 0.27.0
bitsandbytes: 0.50.2
All OK!


## Cell 4: Set paths and mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

WORKSPACE = '/content/drive/MyDrive/LLM4PatchCorrect_workspace'
MODEL_DIR = '/content/drive/MyDrive/LLM4PatchCorrect_model'
OUT_BASE  = '/content/drive/MyDrive/LLM4PatchCorrect_results'
PATCH_COR = f'{WORKSPACE}/LLM4PatchCorrectness/data_checked/patch_cor'
PREMS_DIR = f'{WORKSPACE}/LLM4PatchCorrectness/data_checked/PReMS'

print('Workspace:', os.path.exists(WORKSPACE))
print('Model dir:', os.path.exists(MODEL_DIR))
print('patch_cor:', os.path.exists(PATCH_COR))
print('PReMS dir:', os.path.exists(PREMS_DIR))
print('\nPReMS pkl files:')
for f in sorted(os.listdir(PREMS_DIR)):
    print(f'  {f}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Workspace: True
Model dir: True
patch_cor: True
PReMS dir: True

PReMS pkl files:
  d4c_test_v1_enhanced.pkl
  prism_test_v1_enhanced.pkl
  repairllama_test_v1_enhanced.pkl
  srepair_test_v1_enhanced.pkl
  wang_test_v1_enhanced.pkl


## Cell 5: Copy pkls to patch_cor and patch to 9-tuple

In [ ]:
import pickle, shutil, os

os.makedirs(PATCH_COR, exist_ok=True)

pkl_files = [
    'wang_test_v1_enhanced.pkl',
    'repairllama_test_v1_enhanced.pkl',
    'd4c_test_v1_enhanced.pkl',
    'srepair_test_v1_enhanced.pkl',
    'prism_test_v1_enhanced.pkl',
]

for fname in pkl_files:
    src = os.path.join(PREMS_DIR, fname)
    dst = os.path.join(PATCH_COR, fname)
    shutil.copy(src, dst)
    with open(dst, 'rb') as f:
        data = pickle.load(f)
    if len(data) == 8:
        n = len(data[0])
        data = data + (['']*n,)
        with open(dst, 'wb') as f:
            pickle.dump(data, f)
        print(f'Copied + patched to 9-tuple: {fname} ({n} patches)')
    else:
        print(f'Copied (already 9-tuple):    {fname} ({len(data[0])} patches)')

Copied + patched to 9-tuple: wang_test_v1_enhanced.pkl (421 patches)
Copied (already 9-tuple):    repairllama_test_v1_enhanced.pkl (160 patches)
Copied + patched to 9-tuple: d4c_test_v1_enhanced.pkl (99 patches)
Copied + patched to 9-tuple: srepair_test_v1_enhanced.pkl (158 patches)
Copied + patched to 9-tuple: prism_test_v1_enhanced.pkl (788 patches)


## Cell 6: Create test CSVs and train CSVs

In [ ]:
import pickle, csv, shutil, os

pool_csv = os.path.join(PATCH_COR, 'repairllama_train_cross_v1.csv')
with open(pool_csv, 'r') as f:
    pool_rows = list(csv.reader(f))
print(f'Retrieval pool: {len(pool_rows)} rows')

datasets = {
    'wang':        'wang_test_v1_enhanced.pkl',
    'repairllama': 'repairllama_test_v1_enhanced.pkl',
    'd4c':         'd4c_test_v1_enhanced.pkl',
    'srepair':     'srepair_test_v1_enhanced.pkl',
    'prism':       'prism_test_v1_enhanced.pkl',
}

for name, fname in datasets.items():
    if name != 'repairllama':
        shutil.copy(pool_csv, os.path.join(PATCH_COR, f'{name}_train_cross_v1.csv'))
        print(f'Copied pool -> {name}_train_cross_v1.csv')
    else:
        print(f'Kept original: repairllama_train_cross_v1.csv')

    with open(os.path.join(PATCH_COR, fname), 'rb') as f:
        data = pickle.load(f)
    labels, patches = data[0], data[1]
    with open(os.path.join(PATCH_COR, f'{name}_test_v1.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        for label, patch in zip(labels, patches):
            writer.writerow([label, patch])
    print(f'Written {name}_test_v1.csv ({len(labels)} rows)\n')

Retrieval pool: 3052 rows
Copied pool -> wang_train_cross_v1.csv
Written wang_test_v1.csv (421 rows)

Kept original: repairllama_train_cross_v1.csv
Written repairllama_test_v1.csv (160 rows)

Copied pool -> d4c_train_cross_v1.csv
Written d4c_test_v1.csv (99 rows)

Copied pool -> srepair_train_cross_v1.csv
Written srepair_test_v1.csv (158 rows)

Copied pool -> prism_train_cross_v1.csv
Written prism_test_v1.csv (788 rows)



## Cell 7: Verify all required files exist

In [ ]:
import os

datasets = ['wang', 'repairllama', 'd4c', 'srepair', 'prism']
all_ok = True

for name in datasets:
    files = [
        f'{name}_test_v1_enhanced.pkl',
        f'{name}_test_v1.csv',
        f'{name}_train_cross_v1.csv',
    ]
    print(f'\n=== {name} ===')
    for fname in files:
        path = os.path.join(PATCH_COR, fname)
        exists = os.path.exists(path)
        size = os.path.getsize(path) if exists else 0
        status = 'OK' if exists else 'MISSING'
        print(f'  [{status}] {fname} ({round(size/1024,1)} KB)')
        if not exists:
            all_ok = False

print('\n' + ('All files ready!' if all_ok else 'MISSING FILES — fix before running inference!'))


=== wang ===
  [OK] wang_test_v1_enhanced.pkl (204.9 KB)
  [OK] wang_test_v1.csv (194.5 KB)
  [OK] wang_train_cross_v1.csv (1527.6 KB)

=== repairllama ===
  [OK] repairllama_test_v1_enhanced.pkl (169.4 KB)
  [OK] repairllama_test_v1.csv (165.7 KB)
  [OK] repairllama_train_cross_v1.csv (1527.6 KB)

=== d4c ===
  [OK] d4c_test_v1_enhanced.pkl (173.5 KB)
  [OK] d4c_test_v1.csv (170.9 KB)
  [OK] d4c_train_cross_v1.csv (1527.6 KB)

=== srepair ===
  [OK] srepair_test_v1_enhanced.pkl (1127.9 KB)
  [OK] srepair_test_v1.csv (1127.8 KB)
  [OK] srepair_train_cross_v1.csv (1527.6 KB)

=== prism ===
  [OK] prism_test_v1_enhanced.pkl (341.1 KB)
  [OK] prism_test_v1.csv (318.8 KB)
  [OK] prism_train_cross_v1.csv (1527.6 KB)

All files ready!


## Cell 8: Run inference on all 5 datasets
Automatically skips datasets already done. Results saved to `LLM4PatchCorrect_results/logits_<dataset>/`

In [ ]:
import pickle, os, shutil

patch_cor = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor'
prems_dir = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/PReMS'

for path in [
    os.path.join(patch_cor, 'repairllama_test_v1_enhanced.pkl'),
    os.path.join(prems_dir, 'repairllama_test_v1_enhanced.pkl'),
]:
    with open(path, 'rb') as f:
        data = pickle.load(f)
    if len(data) == 8:
        n = len(data[0])
        data = data + (['']*n,)
        with open(path, 'wb') as f:
            pickle.dump(data, f)
        print(f'Patched to 9-tuple: {path} ({n} patches)')
    else:
        print(f'Already 9-tuple: {path}')

Already 9-tuple: /content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor/repairllama_test_v1_enhanced.pkl
Already 9-tuple: /content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/PReMS/repairllama_test_v1_enhanced.pkl


In [ ]:
import pickle, csv, os, shutil
os.chdir(f'{WORKSPACE}/LLM4PatchCorrectness')

patch_cor = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor'
prems_dir = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/PReMS'
MODEL_DIR = '/content/drive/MyDrive/LLM4PatchCorrect_model'

RESULT_DIR  = f'{MODEL_DIR}/direct/patch_repairllama'
RESULT_FILE = 'test-template=0-topk_example=10-sim_threshold=0.9-max_length=4000-bug-trace-testcase-coverage-similar.pkl'

datasets = ['repairllama', 'wang', 'd4c', 'srepair', 'prism']

with open(os.path.join(patch_cor, 'repairllama_test_v1_enhanced.pkl'), 'rb') as f:
    rl_data = pickle.load(f)
rl_labels, rl_patches = rl_data[0], rl_data[1]

def restore_repairllama():
    with open(os.path.join(patch_cor, 'repairllama_test_v1.csv'), 'w', newline='') as f:
        writer = csv.writer(f)
        for label, patch in zip(rl_labels, rl_patches):
            writer.writerow([label, patch])
    shutil.copy(
        os.path.join(prems_dir, 'repairllama_test_v1_enhanced.pkl'),
        os.path.join(patch_cor, 'repairllama_test_v1_enhanced.pkl')
    )

restore_repairllama()
print(f'Repairllama files restored: {len(rl_labels)} rows')

print('\n=== Running inference ===')
for name in datasets:
    out_dir = f'{MODEL_DIR}/direct/patch_repairllama_{name}'
    os.makedirs(out_dir, exist_ok=True)

    result_file = os.path.join(out_dir, RESULT_FILE)
    if os.path.exists(result_file):
        print(f'SKIPPING {name} — results already exist')
        continue

    if name != 'repairllama':
        shutil.copy(
            os.path.join(patch_cor, f'{name}_test_v1.csv'),
            os.path.join(patch_cor, 'repairllama_test_v1.csv')
        )
        shutil.copy(
            os.path.join(patch_cor, f'{name}_test_v1_enhanced.pkl'),
            os.path.join(patch_cor, 'repairllama_test_v1_enhanced.pkl')
        )

    print(f'\n{"="*50}')
    print(f'Running: {name}')
    print(f'{"="*50}')

    !conda run -n llm4 python main.py \
        --task patch_repairllama \
        --data_dir data_checked \
        --split test \
        --out_dir {out_dir} \
        --gpt2 {MODEL_DIR} \
        --batch_size 1 \
        --do_zeroshot \
        --k 10 \
        --max_length 4000 \
        --n_template 0 \
        --top_k_example 10 \
        --sim_threshold 0.9 \
        --enhancement_option bug-trace-testcase-coverage-similar

    model_result = os.path.join(RESULT_DIR, RESULT_FILE)
    if os.path.exists(model_result):
        shutil.copy(model_result, result_file)
        os.remove(model_result)
        print(f'Saved result: {name}')

    restore_repairllama()
    print(f'Done: {name}')

print('\nAll datasets complete!')

Repairllama files restored: 160 rows

=== Running inference ===
SKIPPING repairllama — results already exist
SKIPPING wang — results already exist

Running: d4c
[2026-10-15 16:21:24,369] [INFO] [real_accelerator.py:110:get_accelerator] Setting ds_accelerator to cuda (auto detect)
Time taken: 1307.3180329799652
/usr/local/envs/llm4/lib/python3.10/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(
/usr/local/envs/llm4/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:49: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  def forward(ctx, input, weight, bias=None):
/usr/local/envs/llm4/lib/python3.10/site-packages/deepspeed/runtime/zero/linear.py:67: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` inst

## Cell 9: Compute AUC and MCC for all datasets

In [ ]:
import pickle, os
from sklearn.metrics import roc_auc_score, matthews_corrcoef

MODEL_DIR = '/content/drive/MyDrive/LLM4PatchCorrect_model'
PATCH_COR = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor'
RESULT_FILE = 'test-template=0-topk_example=10-sim_threshold=0.9-max_length=4000-bug-trace-testcase-coverage-similar.pkl'

datasets = ['repairllama', 'wang', 'd4c', 'srepair', 'prism']

print(f'{"Dataset":<15} {"N":>6} {"Correct":>8} {"Overfit":>8} {"AUC":>8} {"MCC":>8}')
print('-' * 60)

for name in datasets:
    result_path = os.path.join(MODEL_DIR, 'direct', f'patch_repairllama_{name}', RESULT_FILE)
    if not os.path.exists(result_path):
        print(f'{name:<15} NOT FOUND')
        continue

    with open(result_path, 'rb') as f:
        results = pickle.load(f)

    with open(os.path.join(PATCH_COR, f'{name}_test_v1_enhanced.pkl'), 'rb') as f:
        data = pickle.load(f)
    true_labels = data[0]

    correct_losses = results[0]
    wrong_losses   = results[1]

    scores_for_auc = [w - c for c, w in zip(correct_losses, wrong_losses)]

    preds = [1 if c < w else 0 for c, w in zip(correct_losses, wrong_losses)]

    n_correct = sum(true_labels)
    n_overfit = len(true_labels) - n_correct

    try:
        auc = roc_auc_score(true_labels, scores_for_auc)
        mcc = matthews_corrcoef(true_labels, preds)
        print(f'{name:<15} {len(true_labels):>6} {n_correct:>8} {n_overfit:>8} {auc:>8.3f} {mcc:>8.3f}')
    except Exception as e:
        print(f'{name:<15} ERROR: {e}')

Dataset              N  Correct  Overfit      AUC      MCC
------------------------------------------------------------
repairllama        160       63       97    0.592    0.105
wang               421       77      344    0.997    0.939
d4c                 99       86       13    0.528   -0.028
srepair            158       33      125    0.531    0.215
prism              788      137      651    0.763    0.307


In [ ]:
import pickle, os
from sklearn.metrics import roc_auc_score, matthews_corrcoef

MODEL_DIR = '/content/drive/MyDrive/LLM4PatchCorrect_model'
PATCH_COR = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor'
RESULT_FILE = 'test-template=0-topk_example=10-sim_threshold=0.9-max_length=4000-bug-trace-testcase-coverage-similar.pkl'

datasets = ['repairllama', 'wang', 'd4c', 'srepair', 'prism']

print(f'{"Dataset":<15} {"N":>6} {"Correct":>8} {"Overfit":>8} {"AUC":>8} {"MCC":>8}')
print('-' * 60)

for name in datasets:
    result_path = os.path.join(MODEL_DIR, 'direct', f'patch_repairllama_{name}', RESULT_FILE)
    if not os.path.exists(result_path):
        print(f'{name:<15} NOT FOUND')
        continue

    with open(result_path, 'rb') as f:
        results = pickle.load(f)

    with open(os.path.join(PATCH_COR, f'{name}_test_v1_enhanced.pkl'), 'rb') as f:
        data = pickle.load(f)
    true_labels = data[0]

    correct_losses = results[0][:len(true_labels)]
    wrong_losses   = results[1][:len(true_labels)]

    scores_for_auc = [w - c for c, w in zip(correct_losses, wrong_losses)]
    preds = [1 if c < w else 0 for c, w in zip(correct_losses, wrong_losses)]

    n_correct = sum(true_labels)
    n_overfit = len(true_labels) - n_correct

    try:
        auc = roc_auc_score(true_labels, scores_for_auc)
        mcc = matthews_corrcoef(true_labels, preds)
        print(f'{name:<15} {len(true_labels):>6} {n_correct:>8} {n_overfit:>8} {auc:>8.3f} {mcc:>8.3f}')
    except Exception as e:
        print(f'{name:<15} ERROR: {e}')

Dataset              N  Correct  Overfit      AUC      MCC
------------------------------------------------------------
repairllama        160       63       97    0.592    0.105
wang               421       77      344    0.997    0.939
d4c                 99       86       13    0.528   -0.028
srepair            158       33      125    0.531    0.215
prism              788      137      651    0.763    0.307


In [ ]:
import pickle, csv, os

MODEL_DIR = '/content/drive/MyDrive/LLM4PatchCorrect_model'
PATCH_COR = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor'
RESULT_FILE = 'test-template=0-topk_example=10-sim_threshold=0.9-max_length=4000-bug-trace-testcase-coverage-similar.pkl'
OUT_CSV_DIR = '/content/drive/MyDrive/LLM4PatchCorrect_results/csv'
os.makedirs(OUT_CSV_DIR, exist_ok=True)

datasets = ['repairllama', 'wang', 'd4c', 'srepair', 'prism']

for name in datasets:
    result_path = os.path.join(MODEL_DIR, 'direct', f'patch_repairllama_{name}', RESULT_FILE)
    pkl_path    = os.path.join(PATCH_COR, f'{name}_test_v1_enhanced.pkl')

    with open(result_path, 'rb') as f:
        results = pickle.load(f)
    with open(pkl_path, 'rb') as f:
        data = pickle.load(f)

    true_labels    = data[0]
    bug_ids        = data[2]
    correct_losses = results[0][:len(true_labels)]
    wrong_losses   = results[1][:len(true_labels)]

    csv_path = os.path.join(OUT_CSV_DIR, f'{name}_results.csv')
    with open(csv_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['bug_id', 'true_label', 'loss_correct', 'loss_wrong', 'predicted_label', 'score'])
        for label, bug_id, lc, lw in zip(true_labels, bug_ids, correct_losses, wrong_losses):
            pred  = 1 if lc < lw else 0
            score = lw - lc
            writer.writerow([bug_id, label, lc, lw, pred, score])

    print(f'Saved: {name}_results.csv ({len(true_labels)} rows)')

print('\nAll CSVs saved to Drive!')

Saved: repairllama_results.csv (160 rows)
Saved: wang_results.csv (421 rows)
Saved: d4c_results.csv (99 rows)
Saved: srepair_results.csv (158 rows)
Saved: prism_results.csv (788 rows)

All CSVs saved to Drive!


In [ ]:
import pickle, os

PATCH_COR = '/content/drive/MyDrive/LLM4PatchCorrect_workspace/LLM4PatchCorrectness/data_checked/patch_cor'

with open(os.path.join(PATCH_COR, 'wang_test_v1_enhanced.pkl'), 'rb') as f:
    data = pickle.load(f)

print(f'Tuple length: {len(data)}')
print(f'[0] labels:          {len(data[0])} items — example: {data[0][:3]}')
print(f'[1] patches:         {len(data[1])} items — example: {str(data[1][0])[:50]}')
print(f'[2] bug_ids:         {len(data[2])} items — example: {data[2][:3]}')
print(f'[3] fail_test_names: {len(data[3])} items — example: {data[3][0][:2]}')
print(f'[4] fail_test_src:   {len(data[4])} items — example: {list(data[4][0].keys())[:2]}')
print(f'[5] coverages:       {len(data[5])} items — example: {str(data[5][0])[:50]}')
print(f'[6] traces_short:    {len(data[6])} items — example: {str(data[6][0])[:50]}')
print(f'[7] traces_full:     {len(data[7])} items — example: {str(data[7][0])[:50]}')
print(f'[8] dummy:           {len(data[8])} items')

Tuple length: 9
[0] labels:          421 items — example: [1, 0, 1]
[1] patches:         421 items — example: --- +++ @@ -1,13 +1,7 @@         boolean wasWhite=
[2] bug_ids:         421 items — example: ['Lang-10', 'Lang-10', 'Lang-10']
[3] fail_test_names: 421 items — example: []
[4] fail_test_src:   421 items — example: []
[5] coverages:       421 items — example: 
[6] traces_short:    421 items — example: []
[7] traces_full:     421 items — example: []
[8] dummy:           421 items


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/LLM4PatchCorrect_results/csv/wang_results.csv')

print('Label distribution (true):')
print(df['true_label'].value_counts())

print('\nPrediction distribution:')
print(df['predicted_label'].value_counts())

print('\nScore stats for correct patches (true_label=1):')
print(df[df['true_label']==1]['score'].describe())

print('\nScore stats for overfitting patches (true_label=0):')
print(df[df['true_label']==0]['score'].describe())

Label distribution (true):
true_label
0    344
1     77
Name: count, dtype: int64

Prediction distribution:
predicted_label
0    338
1     83
Name: count, dtype: int64

Score stats for correct patches (true_label=1):
count    77.000000
mean      2.792673
std       1.530508
min      -0.718750
25%       1.547363
50%       2.968994
75%       4.046555
max       5.438362
Name: score, dtype: float64

Score stats for overfitting patches (true_label=0):
count    344.000000
mean      -2.949834
std        1.676017
min       -7.553505
25%       -3.827778
50%       -2.843262
75%       -1.844116
max        2.578796
Name: score, dtype: float64
